# Nairobi OS Quickstart: The Sovereign API

Welcome to the **Heavy Iron**. This notebook demonstrates the v0.3.5 "Frictionless" API refit of Nairobi OS. 

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KevinKenya/nairobi-connector-open-source/blob/main/nairobi-benchmarks/nairobiOsBenchmarks.ipynb)

Nairobi OS is a distributed microservice architecture designed for high-performance, zero-copy data analysis. By offloading heavy lifting to a specialized Rust-based refinery daemon and using `memfd` handles, we achieve performance that makes Pandas look like a toy.

## 🚀 Why Nairobi OS?

- **Zero-Copy Ingestion**: Data stays in kernel memory (`memfd`).
- **Rust-Powered Analytics**: All statistical operations are executed in parallelized Rust.
- **Lagos Visual Cortex**: Hardware-accelerated plotting directly from shared memory.

### 🛠️ Google Colab Setup (Run this first if on Colab)

Nairobi OS requires a D-Bus session and the Rust binaries to be built. The following cell handles the environment setup for headless Colab instances.

In [ ]:
import sys
import os
import subprocess

if 'google.colab' in sys.modules:
    print("⚙️ Detected Google Colab. Initializing Heavy Iron environment...")
    
    # 1. Install D-Bus dependencies
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "dbus-x11"], check=True)
    
    # 2. Launch D-Bus Session
    dbus_output = subprocess.check_output(["dbus-launch"]).decode()
    for line in dbus_output.splitlines():
        if "=" in line:
            key, value = line.split("=", 1)
            os.environ[key] = value
            
    # 3. Build & Install Nairobi OS (Assuming repo is cloned or we are in it)
    if not os.path.exists("build_wheel.sh"):
        print("📥 Cloning Nairobi OS Repository...")
        subprocess.run(["git", "clone", "https://github.com/KevinKenya/nairobi-connector-open-source.git"], check=True)
        os.chdir("nairobi-connector-open-source")
    
    print("🏗️ Forging the Heavy Iron (Building Wheel)... This may take 2-3 minutes.")
    subprocess.run(["./build_wheel.sh"], check=True)
    
    print("📦 Installing Nairobi OS...")
    subprocess.run(["pip", "install", "dist/nairobi_os-0.3.5-py3-none-any.whl"], check=True)
    
    print("✅ Colab Environment Ready.")
else:
    print("✅ Native Linux/WSL2 environment detected.")

### 1. The Ignition

In Nairobi OS, we don't just 'import' data; we ignite the refinery. The `connect()` function (a semantic alias for `start_refinery`) ensures the D-Bus session is active and the Axum Refinery daemon is ready for action.

In [ ]:
import nairobi_os as nb
import kagglehub
import os

# Ignite the Refinery
nb.connect()

### 2. The Ingestion

We'll use the NBA Player Statistics dataset. `nb.read_csv()` returns a `SovereignFrame`, which is a high-level wrapper around a Rust `memfd` handle.

In [ ]:
# Download NBA dataset
dataset_path = kagglehub.dataset_download('vivovinco/nba-player-stats')
csv_file = os.path.join(dataset_path, '2021-2022 NBA Player Stats - Regular.csv')

# Ingest into the Sovereign Frame
df = nb.read_csv(csv_file)
print(f"Sovereign Handle ID: {df.handle_id}")

### 3. The Forensic Audit (`.crunch()`)

The `.crunch()` method performs a deep statistical analysis of a column. This happens entirely in Rust, returning a native Python dictionary with zero measurable latency.

In [ ]:
# Analyze points per game
pts_stats = df.crunch("PTS")

print("--- NBA Points Statistics ---")
for key, value in pts_stats.items():
    print(f"{key.capitalize()}: {value:.4f}")

### 4. The Relational Strike (`.correlate()`)

Calculating correlation matrices for large datasets can be slow in Python. Nairobi OS executes this on the metal.

In [ ]:
# Correlate Points, Assists, and Rebounds
corr_matrix = df.correlate("PTS,AST,TRB")

print("--- Correlation Matrix ---")
import json
print(json.dumps(corr_matrix, indent=2))

### 5. The Distillation (`.query()`)

Sometimes you only need a subset of the iron. `.query()` executes SQL directly on the `memfd` and returns a **new** `SovereignFrame` containing only the distilled data.

In [ ]:
# Extract only the points for high-performance visualization
distilled_df = df.query("SELECT PTS FROM dataset")
print(f"New Sovereign Handle: {distilled_df.handle_id}")

### 6. The Visual Cortex (`.plot()`)

Lagos Vision maps the `memfd` directly into the GPU pipeline. No data ever passes through the Python interpreter during rendering.

In [ ]:
# Render the distilled points
distilled_df.plot(width=800, height=400)

### 7. Shutdown

Clean up the refinery when you're done.

In [ ]:
nb.stop_refinery()